<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/19_neural_networks/19_0_PyTorch_Basics/19_0_3_Gradient_Descent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PyTorch Basics: Part 3
## Gradient Descent from Scratch

---

## What This Notebook Is About

In Unit 17, notebook 17_0_5, you plotted RSS against slope values for a fixed intercept and got a U-shaped bowl. The bottom of the bowl is the best slope. To find that bottom, you used a closed-form formula — a direct mathematical shortcut that goes straight from data to the optimal answer.

That shortcut only works because linear regression has a special mathematical structure. Neural networks do not have that structure. There is no formula for the optimal weights of a 10-layer network. The only way to find them is through an iterative search that takes small steps downhill until it reaches the bottom.

That search is **gradient descent**. In the last two notebooks, you learned how PyTorch computes the slope at any point in the loss landscape: `requires_grad=True` builds the computation graph, and `.backward()` computes the gradient. In this notebook, you will put those two tools together and write the training loop from scratch — the same loop that runs inside every neural network.

We will use the flipper-to-body-mass regression from Unit 17 so that the dataset is not a distraction. The data is familiar; the mechanism is entirely new.

**What you will learn:**
1. How to set up the loss landscape and see the bowl from Unit 17 in PyTorch
2. How to take a single gradient descent step by hand — and verify the loss decreased
3. How to wrap that step in a loop and watch the loss converge
4. How gradient descent finds essentially the same line as sklearn's analytic solution
5. Why the learning rate is the most consequential hyperparameter to get right

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import seaborn as sns
from sklearn.linear_model import LinearRegression

sns.set_theme(style='whitegrid')
torch.manual_seed(42)

# Load the same penguins data from Unit 17
penguins = sns.load_dataset('penguins').dropna().reset_index(drop=True)

flipper_raw  = torch.tensor(penguins['flipper_length_mm'].values, dtype=torch.float32)
bodymass_raw = torch.tensor(penguins['body_mass_g'].values,       dtype=torch.float32)

# Standardize both variables (zero mean, unit variance).
# This is the same StandardScaler preprocessing you used in sklearn Pipelines.
# It keeps the gradients at a manageable scale so we can use a sensible learning rate.
x_mean, x_std = flipper_raw.mean(), flipper_raw.std()
y_mean, y_std = bodymass_raw.mean(), bodymass_raw.std()

x = (flipper_raw  - x_mean) / x_std   # standardized flipper length
y = (bodymass_raw - y_mean) / y_std   # standardized body mass

print(f'Dataset: {len(x)} penguins after dropping missing values')
print(f'Flipper length  — mean: {x_mean.item():.1f} mm,  std: {x_std.item():.1f} mm')
print(f'Body mass       — mean: {y_mean.item():.0f} g,   std: {y_std.item():.0f} g')
print()
print('After standardizing:')
print(f'  x  mean ≈ {x.mean().item():.2e},  std ≈ {x.std().item():.4f}')
print(f'  y  mean ≈ {y.mean().item():.2e},  std ≈ {y.std().item():.4f}')

In [ ]:
# The same scatter you have seen since Unit 17
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(flipper_raw.numpy(), bodymass_raw.numpy(),
           alpha=0.55, edgecolors='white', s=55, color='steelblue')
ax.set_xlabel('Flipper Length (mm)')
ax.set_ylabel('Body Mass (g)')
ax.set_title('Palmer Penguins: Flipper vs. Body Mass')
plt.tight_layout()
plt.show()

r = torch.corrcoef(torch.stack([flipper_raw, bodymass_raw]))[0, 1].item()
print(f'Pearson r = {r:.3f}  — strong positive linear relationship.')
print('For standardized variables, the optimal regression slope equals r exactly.')

---

## Section 1: The Loss Bowl

In 17_0_5 you plotted RSS against one parameter — the slope — with the intercept fixed. The result was a U-shaped bowl, and the bottom was the best slope.

Let's reproduce that visualization in PyTorch, now using MSE (mean squared error, which is RSS divided by n) and standardized variables. We will sweep through many values of the weight w, compute the MSE at each one, and plot the bowl. The bias is held at 0, which is the optimal value for centered data — when both x and y have mean 0, the best-fit line passes through the origin.

After the plot, we will start gradient descent at w = 0 — the far left side of the bowl — and watch it roll toward the bottom.

In [ ]:
# Sweep w from -0.5 to 2.0, compute MSE at each value with bias fixed at 0
w_range  = torch.linspace(-0.5, 2.0, 300)
mse_bowl = []

with torch.no_grad():
    for w_val in w_range:
        pred = w_val * x          # bias = 0
        mse_bowl.append(((pred - y) ** 2).mean().item())

min_idx = int(np.argmin(mse_bowl))
w_min   = w_range[min_idx].item()
w0_idx  = int((w_range - 0.0).abs().argmin())

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(w_range.numpy(), mse_bowl, 'steelblue', lw=2.5)
ax.scatter([0.0],   [mse_bowl[w0_idx]],  color='#C0392B', s=130, zorder=5,
           label='starting point  (w = 0)')
ax.scatter([w_min], [mse_bowl[min_idx]], color='#27AE60', s=130, zorder=5,
           label=f'minimum  (w ≈ {w_min:.3f})')
ax.set_xlabel('weight  w')
ax.set_ylabel('MSE loss')
ax.set_title('The Loss Bowl  (bias = 0, both variables standardized)')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Bowl minimum at w ≈ {w_min:.3f}')
print(f'Pearson r         = {r:.3f}  (they match: optimal w = r for standardized data)')
print()
print(f'MSE at starting point (w=0): {mse_bowl[w0_idx]:.4f}')
print(f'MSE at minimum (w≈{w_min:.3f}):   {mse_bowl[min_idx]:.4f}')
print(f'That gap is what gradient descent will close.')

---

## Section 2: One Step by Hand

Gradient descent works like this: look at the slope of the bowl at your current position, take a small step in the downhill direction, repeat.

The slope at any point is the gradient — exactly what `.backward()` computes. The "small step" is the learning rate times the gradient, subtracted from the current parameter value.

Let's do one step explicitly before wrapping it in a loop, so every line has an obvious purpose.

In [ ]:
# Start at a deliberately bad position
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

lr = 0.1

# --- Forward pass: compute predictions and loss ---
pred         = w * x + b
loss_before  = ((pred - y) ** 2).mean()

print('Before the step:')
print(f'  w = {w.item():.4f},  b = {b.item():.4f}')
print(f'  loss = {loss_before.item():.4f}')
print()

# --- Backward pass: compute the slope at this point ---
loss_before.backward()

print('Gradients (slope of the bowl at this point):')
print(f'  dLoss/dw = {w.grad.item():.4f}  ← negative means loss falls as w increases')
print(f'  dLoss/db = {b.grad.item():.4f}  ← near zero because y is already centered')
print()

# --- Parameter update: step downhill ---
with torch.no_grad():
    w -= lr * w.grad   # w moves in the direction that reduces loss
    b -= lr * b.grad
w.grad.zero_()
b.grad.zero_()

# --- Verify: did the loss go down? ---
with torch.no_grad():
    loss_after = ((w * x + b - y) ** 2).mean()

print('After the step:')
print(f'  w = {w.item():.4f},  b = {b.item():.4f}')
print(f'  loss = {loss_after.item():.4f}')
print(f'  improvement = {loss_before.item() - loss_after.item():.4f}')

---

## Section 3: The Training Loop

One step moves us a little closer to the minimum. We need to repeat the process until the loss stops improving meaningfully. That loop is the training algorithm.

Every gradient descent training loop has the same five-line structure:

1. Zero the gradients from the last step
2. Compute predictions (forward pass — builds the graph)
3. Compute the loss
4. Call `.backward()` (backward pass — fills in the gradients)
5. Update the parameters (step downhill)

This is the exact same structure as the training loop for a 100-layer neural network. The only thing that changes is what happens inside step 2 — the model's forward pass gets more complex, but the outer loop is identical.

In [ ]:
# Re-initialize from scratch
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

lr       = 0.1
n_epochs = 300
history  = []

for epoch in range(n_epochs):
    # 1. Zero gradients from last step
    if w.grad is not None: w.grad.zero_()
    if b.grad is not None: b.grad.zero_()

    # 2 & 3. Forward pass + loss
    pred = w * x + b
    loss = ((pred - y) ** 2).mean()

    # 4. Backward pass
    loss.backward()

    # 5. Parameter update
    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad

    history.append(loss.item())

# Plot the loss curve
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history, 'steelblue', lw=2)
ax.axhline(y=history[-1], color='#27AE60', linestyle='--', alpha=0.8,
           label=f'final loss = {history[-1]:.4f}')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE loss (standardized units)')
ax.set_title('Loss Curve over 300 Epochs  (lr = 0.1)')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Starting loss : {history[0]:.4f}')
print(f'Final loss    : {history[-1]:.4f}')
print()
print(f'Converged to  : w = {w.item():.4f},  b = {b.item():.4f}')
print(f'Optimal slope is Pearson r = {r:.4f}  — they agree.')

---

## Section 4: Does It Find the Same Line as sklearn?

Gradient descent is an iterative approximation. The analytic solution that sklearn uses is exact (for linear regression). Let's check how close they are, and then draw both lines on the original scatter to see whether the difference is visible.

In [ ]:
# Analytic solution via sklearn on the original (unstandardized) data
sk = LinearRegression()
sk.fit(flipper_raw.numpy().reshape(-1, 1), bodymass_raw.numpy())

# Convert gradient descent result from standardized back to original units.
# In standardized space: pred_std = w * x_std + b
# In original space:     pred_g   = y_mean + y_std * pred_std
#                                 = y_mean + y_std*(w*(x_mm - x_mean)/x_std + b)
# So: slope_orig     = y_std * w / x_std
#     intercept_orig = y_mean - slope_orig * x_mean + y_std * b
with torch.no_grad():
    slope_gd     = (w * y_std / x_std).item()
    intercept_gd = (y_mean + y_std * b).item() - slope_gd * x_mean.item()

print('Parameters in original units:')
print(f'  sklearn  — slope: {sk.coef_[0]:.3f} g/mm,  intercept: {sk.intercept_:.1f} g')
print(f'  GD       — slope: {slope_gd:.3f} g/mm,  intercept: {intercept_gd:.1f} g')
print()
print(f'  Difference in slope    : {abs(sk.coef_[0] - slope_gd):.4f} g/mm')
print(f'  Difference in intercept: {abs(sk.intercept_ - intercept_gd):.1f} g')
print()

# Draw both lines on the original scatter
x_line  = np.linspace(flipper_raw.min().item() - 3, flipper_raw.max().item() + 3, 100)
y_sk    = sk.coef_[0]  * x_line + sk.intercept_
y_gd_ln = slope_gd     * x_line + intercept_gd

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(flipper_raw.numpy(), bodymass_raw.numpy(),
           alpha=0.5, edgecolors='white', s=50, color='steelblue', label='data')
ax.plot(x_line, y_sk,    color='#27AE60', lw=3.0, linestyle='--',
        label='sklearn  (analytic)', zorder=4)
ax.plot(x_line, y_gd_ln, color='#C0392B', lw=2.0, linestyle=':',
        label='gradient descent  (300 epochs)', zorder=5)
ax.set_xlabel('Flipper Length (mm)')
ax.set_ylabel('Body Mass (g)')
ax.set_title('Gradient Descent vs. Analytic Solution')
ax.legend()
plt.tight_layout()
plt.show()

The two lines are indistinguishable. Gradient descent converges to the same answer as the exact analytic formula — it just takes more computation to get there.

For linear regression, the analytic shortcut is always preferable. But for neural networks, no such shortcut exists. Gradient descent — the iterative algorithm you just implemented — is the only option. That is why understanding it matters.

---

## Section 5: The Learning Rate

The learning rate is the size of each step. It is the most consequential hyperparameter in any gradient descent optimizer.

You might think it hardly matters — just pick something small and the algorithm will converge eventually. Let's see what actually happens across three choices.

- **Too small:** the algorithm converges, but so slowly that you would need far more steps than are practical
- **Just right:** the algorithm converges cleanly within a reasonable number of steps
- **Too large:** the algorithm overshoots the minimum on the first step, bounces to the other side, overshoots again with an even larger step, and diverges — the loss explodes instead of falling

In [ ]:
def run_gd(lr_, n_epochs=300, diverge_threshold=50):
    w_ = torch.tensor(0.0, requires_grad=True)
    b_ = torch.tensor(0.0, requires_grad=True)
    losses_ = []
    for _ in range(n_epochs):
        if w_.grad is not None: w_.grad.zero_()
        if b_.grad is not None: b_.grad.zero_()
        pred_  = w_ * x + b_
        loss_  = ((pred_ - y) ** 2).mean()
        val    = loss_.item()
        if not np.isfinite(val) or val > diverge_threshold:
            losses_ += [float('nan')] * (n_epochs - len(losses_))
            break
        loss_.backward()
        with torch.no_grad():
            w_ -= lr_ * w_.grad
            b_ -= lr_ * b_.grad
        losses_.append(val)
    return losses_

configs = [
    (0.001, 'lr = 0.001   (too small)',  '#DD8452'),
    (0.100, 'lr = 0.1     (just right)', '#4C72B0'),
    (1.500, 'lr = 1.5     (too large)',  '#C44E52'),
]

fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)

for ax, (lr_, label, color) in zip(axes, configs):
    hist   = run_gd(lr_)
    finite = [v for v in hist if np.isfinite(v)]
    ax.plot(range(300), hist, color=color, lw=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE loss')
    ax.set_title(label, fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3)
    if finite:
        ax.set_ylim(bottom=-0.02)
    has_nan = any(not np.isfinite(v) for v in hist)
    if has_nan and finite:
        ax.text(len(finite) + 20, max(finite) * 0.35,
                f'diverged after\n{len(finite)} steps',
                ha='left', fontsize=9, color=color, style='italic')

plt.suptitle('Learning Rate Sensitivity', fontsize=13, fontweight='bold')
plt.show()

print('lr = 0.001  Still converging after 300 steps — roughly halfway to the minimum.')
print('lr = 0.1    Converges cleanly. Loss reaches the minimum within ~50 epochs.')
print('lr = 1.5    Overshoots immediately. Loss diverges after 3-4 steps.')
print()
print('There is no universal formula for the right learning rate.')
print('The standard approach: try a few values spanning several orders of magnitude')
print('(e.g., 0.001, 0.01, 0.1) and pick the one that converges cleanly.')

---

## Putting It All Together

| Concept | What it means |
|---|---|
| Loss landscape | A surface showing the loss at every possible parameter value; gradient descent navigates this surface |
| The training loop | Zero gradients → forward pass → compute loss → `.backward()` → update parameters; repeated every epoch |
| Epoch | One complete pass through the loop; 300 epochs = 300 parameter updates |
| Learning rate | The step size; controls how aggressively parameters are updated each epoch |
| Convergence | When the loss stops meaningfully decreasing; the parameters have found (approximately) the bowl's bottom |
| Analytic vs. iterative | Linear regression has a closed-form solution; neural networks do not — gradient descent is the only option |

**The three things standardization bought us:** (1) both gradients are on a similar scale, so a single learning rate works for both w and b; (2) the loss starts near 1.0 regardless of the original units; (3) the optimal bias is exactly 0, which simplifies the visualization. In a real pipeline you would use `StandardScaler` inside a sklearn `Pipeline` — or PyTorch's own preprocessing tools.

**What is coming next:** In the next sub-module you will build actual neural networks using `nn.Module` and apply this exact training loop to them. The loop itself will not change — only the forward pass gets more complex as the model gains layers and activation functions.